<a href="https://colab.research.google.com/github/sophiastmn/ML_Labs/blob/main/NeuralNetworks2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
############################################################# Problem #1 ###############################################################
import random;
import matplotlib.pyplot as plot;
import time;
import math;

class NeuralNetwork:
  perceptrons = [];

  def createPerceptrons(self, sizes, points, wholeSize):
    #creates empty perceptrons where size = [# perceptrons in input layer, # perceptrons in hidden layer #1, ... # perceptrons in hidden layer #n, # perceptrons in output layer]
    for k, layerSize in enumerate(sizes):
      perceptronLayer = [];
      for s in range(layerSize):
        singlePerceptron = Perceptron();
        singlePerceptron.setWeights([random.random() for i in range(wholeSize[k])]);
        singlePerceptron.setBias(random.random());
        perceptronLayer.append(singlePerceptron);
      self.perceptrons.append(perceptronLayer);

  def setWeights(self, weights):
    #takes in weights as [[weights for input layer], [weights for hidden layer 1], ..., [weights for hidden layer n], [weights for output layer]]
    for i, pLayer in enumerate(self.perceptrons):
      for j, perc in enumerate(pLayer):
        perc.setWeights(weights[i][j]);

  def setBias(self, bias):
    #takes in bias as [[biases for input layer], [biases for hidden layer 1], ..., [biases for hidden layer n], [biases for output layer]]
    for i, pLayer in enumerate(self.perceptrons):
      for j, perc in enumerate(pLayer):
        perc.setBias(bias[i][j]);

  def calculateOutputs(self, point):
    #calculate the "actual" values given input values
    intermittent = [[p for p in point]];

    #for each layer, calculate the intermittent outputs
    for layer in self.perceptrons:
      intermittent.append([perc.returnA(intermittent[-1]) for perc in layer]);

    #we need all of the intermittent values, so return them all
    return intermittent;

  def getWeights(self):
    #get the weights for all of the perceptrons
    weights = [];
    for layer in self.perceptrons:
      weightRow = [];
      for perc in layer:
        pWeights = perc.weights;
        for p in pWeights:
          weightRow.append(p);
      weights.append(weightRow);
    return weights;

  def getBias(self):
    #get the biases for all of the perceptrons
    bias = [];
    for layer in self.perceptrons:
      biasRow = [];
      for perc in layer:
        biasRow.append(perc.bias);
      bias.append(biasRow);
    return bias;

  def findPreErrors(self, aValues, trainingValues, point):
    #get the weights
    newWeights = self.getWeights();

    #find all of the errors
    preErrors = [[] for w in newWeights];
    #the last weights are readjusted with the formula: (a-t)(a)(1-a)
    currLen = len(aValues[-1]);
    preErrors[-1] = [(aValues[-1][i//currLen]-trainingValues[i//currLen])*aValues[-1][i//currLen]*(1-aValues[-1][i//currLen]) for i in range(len(newWeights[-1]))];
    #adjust the rest of the weights with the formula: prevError*weightToNode*a*(1-a)
    for i in range(len(newWeights)-1):
      currLen = len(aValues[-i-2]);
      preErrors[-i-2] = [sum([preE*newWeights[-i-1][k] for k, preE in enumerate(preErrors[-i-1]) if k%currLen == j//currLen])*
                         aValues[-i-2][j//currLen]*(1-aValues[-i-2][j//currLen]) for j in range(len(newWeights[-i-2]))];
    return preErrors;

  def adjustWeights(self, aValues, trainingValues, lv, point):
    #readjust the weights
    #get the weights
    newWeights = self.getWeights();

    #find all of the errors
    preError = self.findPreErrors(aValues, trainingValues, point);
    #actually readjust the weights
    newerWeights = [[lv*preError[i][j]*aValues[i][j//len(aValues[i])]*-1 + newWeights[i][j] for j in range(len(newWeights[i]))] for i in range(len(newWeights))];

    #change the format of the weights to readjust them
    transformedWeights = [];
    for index, layer in enumerate(newerWeights):
      layerWeights = [];
      for i in range(len(aValues[index+1])):
        start = i*len(aValues[index+1]);
        jump = len(aValues[index]);
        layerWeights.append(layer[start:start + jump]);
      transformedWeights.append(layerWeights);
    self.setWeights(transformedWeights);

  def adjustBias(self, aValues, trainingValues, lv, point):
    #readjust the bias
    #get the biases
    biases = self.getBias();

    #find all of the errors
    preError = self.findPreErrors(aValues, trainingValues, point);

    #actually readjust the bias
    newerBias = [[lv*preError[i][j]*-1 + biases[i][j] for j in range(len(biases[i]))] for i in range(len(biases))];
    self.setBias(newerBias);

class Perceptron:
  weights = [];
  bias = 0;

  def setBias(self, val):
    self.bias = val;
  def setWeights(self, newWeights):
    self.weights = newWeights;

  def adjustBias(self, errorRate, learningRate):
    biasNew = self.bias + errorRate*learningRate;
    if biasNew == self.bias:
      return False;
    self.bias = biasNew;
    return True;

  def adjustWeights(self, errorRate, learningRate, pt):
    weightsNew = [w + errorRate*learningRate*pt[i] for i, w in enumerate(self.weights)];
    if weightsNew == self.weights:
      return False;
    self.weights = weightsNew;
    return True;

  def returnA(self, point):
    return self.activationFunction(self.multiply(point) + self.bias);

  def multiply(self, point):
    return sum([self.weights[i] * point[i] for i in range(len(point))]);

  def activationFunction(self, value):
    return 1/(1+math.e**(-1*value));


def trainNetwork(nn, points, trainingValues, lv):
  continue_question = True;
  nn.setWeights([[[0.15, 0.2], [0.25, 0.3]], [[0.4, 0.45], [0.5, 0.55]]]);
  nn.setBias([[0.35, 0.35], [0.6, 0.6]]);
  while continue_question:
    continue_question = epoch(nn, points, trainingValues, lv);

def calculateError(training, actual):
    #calculate the mean squared error using the training and actual values
    sum = 0.0;
    for i in range(len(training)):
      sum += (training[i] - actual[i])**2;
    return sum/len(training);

def epoch(nn, points, trainingValues, lv):
  change = False;
  for p in points:
    a = nn.calculateOutputs(points);
    error = calculateError(trainingValues, a[-1]);
    wChange = nn.adjustWeights(a, trainingValues, lv, p);
    bChange = nn.adjustBias(a, trainingValues, lv, p);
    if error > 0.000002:
      change = True;
  return change;


def main():
  random.seed(1880333);
  #generate a random line
  m = random.random()*3; #get a random slope between 0 and 3
  y = random.random()*6; #get a random line between 0 and 6
  learningVariable = 0.5; #set the learning variable

  #initialize the points
  points = [0.05, 0.10];

  #give the training values
  trainingValues = [0.01, 0.99];

  #make a network
  nn = NeuralNetwork();
  nn.createPerceptrons([2, 2], points, [2, 2, 2]);

  #train the perceptron
  startTime = time.time();
  trainNetwork(nn, points, trainingValues, learningVariable);
  endTime = time.time();
  print(f"Training time: {endTime - startTime}s");


if __name__ == "__main__":
  main();

0.2983711087600027
0.2839318297648284
0.2689364504808065
0.2536293774872803
0.23830139276704046
0.22326417406358048
0.2088171345421957
0.19521399139241158
0.18263745210555765
0.17118782042612973
0.16088642813149512
0.1516902277433973
0.1435116893235778
0.13623860138932067
0.1297503562134884
0.12392944573364101
0.11866840819088553
0.1138731952310445
0.10946407901887245
0.10537506434313447
0.10155251703969155
0.09795347817390881
0.09454394510261802
0.09129727029191825
0.08819274614388223
0.08521439554041396
0.08234996162150984
0.07959007783870534
0.07692759475967847
0.07435703979231521
0.07187418776276987
0.0694757228831064
0.06715897541795271
0.06492171898195051
0.06276201673533767
0.06067810675842184
0.05866831859031074
0.056731014346634204
0.05486454902412809
0.05306724559418605
0.051337381315591034
0.04967318238619283
0.04807282462698878
0.04653443836846348
0.045056116103390935
0.043635921795230306
0.04227190099734614
0.040962091154562744
0.03970453163274522
0.03849727316087182
0.037